In [13]:
checkpoint = "prajjwal1/bert-tiny"
tokenizer_checkpoint = "bert-base-uncased"
dataset_name = "imdb"

from transformers import AutoModel

model = AutoModel.from_pretrained(checkpoint)

from chop.tools import get_tokenized_dataset

dataset, tokenizer = get_tokenized_dataset(
    dataset=dataset_name,
    checkpoint=tokenizer_checkpoint,
    return_tokenizer=True,
)

from pathlib import Path
import dill

with open(f"{Path.home()}/tutorial_5_best_model.pkl", "rb") as f:
    base_model = dill.load(f)

INFO     Tokenizing dataset imdb with AutoTokenizer for bert-base-uncased.


# Task 1

In Tutorial 6, all layers allocated to IntegerLinear are allocated the same width and fractional width. This is suboptimal, as different layers may have different sensitivities to quantization.

-    Modify the code to allow different layers to have widths in the range [8, 16, 32] and fractional widths in the range [2, 4, 8]. Expose this choice as an additional hyperparameter for the Optuna sampler.

-    Run the search again, and plot a figure that has the number of trials on the x axis, and the maximum achieved accuracy up to that point on the y axis.



## Extended Precision Search

This notebook extends the original search to include **all supported precision types** for Linear layers in MASE:

### Supported Precision Types:
1. **float32** - Full precision (baseline)
2. **integer** - Fixed-point integer quantization with configurable width and fractional bits
3. **minifloat_denorm** - Minifloat with denormalized representation
4. **minifloat_ieee** - IEEE-compliant minifloat format
5. **log** - Logarithmic quantization
6. **block_fp** - Block floating point with shared exponents
7. **block_minifloat** - Block minifloat quantization
8. **block_log** - Block logarithmic quantization
9. **binary** - Binary neural networks (1-bit weights)
10. **binary_scaling** - Binary with scaling factors

### Search Space Parameters:
- **Integer**: width ∈ {8, 16, 32}, frac_width ∈ {2, 4, 8}
- **Minifloat**: width ∈ {8, 16}, exponent_width ∈ {3, 4, 5}, exponent_bias ∈ {7, 15}
- **Log**: width ∈ {4, 8}
- **Block-based**: block_size ∈ {8, 16, 32}
- **Binary**: stochastic ∈ {True, False}, bipolar ∈ {True, False}

### Methodology:
1. Run separate Optuna studies for each precision type
2. Each study optimizes layer-specific hyperparameters
3. Compare cumulative maximum accuracy across trials
4. Visualize and analyze results

In [14]:
# Defining Search Space with Working Precision Types

import torch
from chop.nn.quantized.modules.linear import (
    LinearInteger,
    LinearMinifloatDenorm,
    LinearMinifloatIEEE,
    LinearLog,
    LinearBinary,
    LinearBinaryScaling,
)

# ============================================================================
# PRECISION TYPE DEFINITIONS
# Only include precision types that have working gradient implementations
# ============================================================================

PRECISION_TYPES = {
    "float32": {
        "class": torch.nn.Linear,
        "config_generator": None,  # No config needed for full precision
    },
    "integer": {
        "class": LinearInteger,
        "config_generator": lambda trial, name, search_space: {
            "data_in_width": trial.suggest_categorical(f"{name}_int_width", search_space["int_width"]),
            "data_in_frac_width": trial.suggest_categorical(f"{name}_int_frac_width", search_space["frac_width"]),
            "weight_width": trial.suggest_categorical(f"{name}_weight_width", search_space["int_width"]),
            "weight_frac_width": trial.suggest_categorical(f"{name}_weight_frac_width", search_space["frac_width"]),
            "bias_width": trial.suggest_categorical(f"{name}_bias_width", search_space["int_width"]),
            "bias_frac_width": trial.suggest_categorical(f"{name}_bias_frac_width", search_space["frac_width"]),
        },
    },
    "minifloat_denorm": {
        "class": LinearMinifloatDenorm,
        "config_generator": lambda trial, name, search_space: {
            "data_in_width": trial.suggest_categorical(f"{name}_mfd_width", search_space["minifloat_width"]),
            "data_in_exponent_width": trial.suggest_categorical(f"{name}_mfd_exp_width", search_space["exponent_width"]),
            "data_in_exponent_bias": trial.suggest_categorical(f"{name}_mfd_exp_bias", search_space["exponent_bias"]),
            "weight_width": trial.suggest_categorical(f"{name}_mfd_w_width", search_space["minifloat_width"]),
            "weight_exponent_width": trial.suggest_categorical(f"{name}_mfd_w_exp_width", search_space["exponent_width"]),
            "weight_exponent_bias": trial.suggest_categorical(f"{name}_mfd_w_exp_bias", search_space["exponent_bias"]),
            "bias_width": trial.suggest_categorical(f"{name}_mfd_b_width", search_space["minifloat_width"]),
            "bias_exponent_width": trial.suggest_categorical(f"{name}_mfd_b_exp_width", search_space["exponent_width"]),
            "bias_exponent_bias": trial.suggest_categorical(f"{name}_mfd_b_exp_bias", search_space["exponent_bias"]),
        },
    },
    "minifloat_ieee": {
        "class": LinearMinifloatIEEE,
        "config_generator": lambda trial, name, search_space: {
            "data_in_width": trial.suggest_categorical(f"{name}_mfi_width", search_space["minifloat_width"]),
            "data_in_exponent_width": trial.suggest_categorical(f"{name}_mfi_exp_width", search_space["exponent_width"]),
            "data_in_exponent_bias": trial.suggest_categorical(f"{name}_mfi_exp_bias", search_space["exponent_bias"]),
            "weight_width": trial.suggest_categorical(f"{name}_mfi_w_width", search_space["minifloat_width"]),
            "weight_exponent_width": trial.suggest_categorical(f"{name}_mfi_w_exp_width", search_space["exponent_width"]),
            "weight_exponent_bias": trial.suggest_categorical(f"{name}_mfi_w_exp_bias", search_space["exponent_bias"]),
            "bias_width": trial.suggest_categorical(f"{name}_mfi_b_width", search_space["minifloat_width"]),
            "bias_exponent_width": trial.suggest_categorical(f"{name}_mfi_b_exp_width", search_space["exponent_width"]),
            "bias_exponent_bias": trial.suggest_categorical(f"{name}_mfi_b_exp_bias", search_space["exponent_bias"]),
        },
    },
    "log": {
        "class": LinearLog,
        "config_generator": lambda trial, name, search_space: {
            "data_in_width": trial.suggest_categorical(f"{name}_log_width", search_space["log_width"]),
            "data_in_exponent_bias": trial.suggest_categorical(f"{name}_log_exp_bias", search_space["exponent_bias"]),
            "weight_width": trial.suggest_categorical(f"{name}_log_w_width", search_space["log_width"]),
            "weight_exponent_bias": trial.suggest_categorical(f"{name}_log_w_exp_bias", search_space["exponent_bias"]),
            "bias_width": trial.suggest_categorical(f"{name}_log_b_width", search_space["log_width"]),
            "bias_exponent_bias": trial.suggest_categorical(f"{name}_log_b_exp_bias", search_space["exponent_bias"]),
        },
    },
    "binary": {
        "class": LinearBinary,
        "config_generator": lambda trial, name, search_space: {
            "weight_stochastic": trial.suggest_categorical(f"{name}_bin_w_stochastic", [True, False]),
            "weight_bipolar": trial.suggest_categorical(f"{name}_bin_w_bipolar", [True, False]),
        },
    },
    "binary_scaling": {
        "class": LinearBinaryScaling,
        "config_generator": lambda trial, name, search_space: {
            "data_in_stochastic": trial.suggest_categorical(f"{name}_bins_x_stochastic", [True, False]),
            "data_in_bipolar": trial.suggest_categorical(f"{name}_bins_x_bipolar", [True, False]),
            "weight_stochastic": trial.suggest_categorical(f"{name}_bins_w_stochastic", [True, False]),
            "weight_bipolar": trial.suggest_categorical(f"{name}_bins_w_bipolar", [True, False]),
            "bias_stochastic": trial.suggest_categorical(f"{name}_bins_b_stochastic", [True, False]),
            "bias_bipolar": trial.suggest_categorical(f"{name}_bins_b_bipolar", [True, False]),
            "binary_training": True,  # Always enable binary training for search
        },
    },
}

# ============================================================================
# SEARCH SPACE DEFINITION
# Defines the hyperparameter ranges for each precision type
# ============================================================================

search_space = {
    # Integer quantization parameters
    "int_width": [8, 16, 32],
    "frac_width": [2, 4, 8],
    
    # Minifloat parameters (for denorm, IEEE)
    "minifloat_width": [8, 16],
    "exponent_width": [3, 4, 5],
    "exponent_bias": [7, 15],
    
    # Log quantization parameters
    "log_width": [4, 8],
    
    # Available precision types for layer selection
    "precision_types": list(PRECISION_TYPES.keys()),
}

# ============================================================================
# MODEL CONSTRUCTOR
# Creates a model with layers quantized according to Optuna suggestions
# ============================================================================

from chop.tools.utils import deepsetattr
from copy import deepcopy


def construct_model(trial, precision_type=None):
    """
    Construct a model with quantized layers based on Optuna trial suggestions.
    
    Args:
        trial: Optuna trial object for hyperparameter suggestions
        precision_type: If specified, use this precision type for all layers.
                       If None, let Optuna choose per-layer precision.
    
    Returns:
        A copy of base_model with layers replaced according to precision choices
    """
    trial_model = deepcopy(base_model)
    
    for name, layer in trial_model.named_modules():
        if isinstance(layer, torch.nn.Linear):
            # Determine the precision type for this layer
            if precision_type is not None:
                # Use the specified precision type for all layers
                chosen_precision = precision_type
            else:
                # Let Optuna choose the precision type per layer
                chosen_precision = trial.suggest_categorical(
                    f"{name}_precision_type",
                    search_space["precision_types"],
                )
            
            precision_info = PRECISION_TYPES[chosen_precision]
            new_layer_cls = precision_info["class"]
            
            # Skip if full precision (torch.nn.Linear)
            if new_layer_cls == torch.nn.Linear:
                continue
            
            # Build keyword arguments for the new layer
            kwargs = {
                "in_features": layer.in_features,
                "out_features": layer.out_features,
                "bias": layer.bias is not None,
            }
            
            # Generate config using the precision-specific config generator
            config_generator = precision_info["config_generator"]
            if config_generator is not None:
                kwargs["config"] = config_generator(trial, name, search_space)
            
            # Create the new layer and copy weights
            new_layer = new_layer_cls(**kwargs)
            new_layer.weight.data = layer.weight.data.clone()
            if layer.bias is not None and new_layer.bias is not None:
                new_layer.bias.data = layer.bias.data.clone()
            
            # Replace the layer in the model
            deepsetattr(trial_model, name, new_layer)
    
    return trial_model


print("✓ Search space defined with", len(PRECISION_TYPES), "precision types:")
for name in PRECISION_TYPES:
    print(f"  - {name}: {PRECISION_TYPES[name]['class'].__name__}")
print("\nNote: Block-based quantizers (BlockFP, BlockMinifloat, BlockLog) removed due to gradient issues.")

✓ Search space defined with 7 precision types:
  - float32: Linear
  - integer: LinearInteger
  - minifloat_denorm: LinearMinifloatDenorm
  - minifloat_ieee: LinearMinifloatIEEE
  - log: LinearLog
  - binary: LinearBinary
  - binary_scaling: LinearBinaryScaling

Note: Block-based quantizers (BlockFP, BlockMinifloat, BlockLog) removed due to gradient issues.


In [15]:
# Print search space summary
print("=" * 60)
print("SEARCH SPACE SUMMARY")
print("=" * 60)
for key, values in search_space.items():
    print(f"{key}: {values}")

SEARCH SPACE SUMMARY
int_width: [8, 16, 32]
frac_width: [2, 4, 8]
minifloat_width: [8, 16]
exponent_width: [3, 4, 5]
exponent_bias: [7, 15]
log_width: [4, 8]
precision_types: ['float32', 'integer', 'minifloat_denorm', 'minifloat_ieee', 'log', 'binary', 'binary_scaling']


In [16]:
# ============================================================================
# STUDY CONFIGURATION
# Configuration for running precision-specific studies
# ============================================================================

from optuna.samplers import GridSampler, RandomSampler, TPESampler

sampler = RandomSampler()

In [17]:
from chop.tools import get_trainer
import traceback


def objective(trial):
    try:
        # Define the model
        model = construct_model(trial)

        trainer = get_trainer(
            model=model,
            tokenized_dataset=dataset,
            tokenizer=tokenizer,
            evaluate_metric="accuracy",
            num_train_epochs=1,
        )

        trainer.train()
        eval_results = trainer.evaluate()

        # Store precision type used
        precision_used = trial.params.get(f"{list(trial.params.keys())[0].split('_precision_type')[0]}_precision_type", "unknown")
        trial.set_user_attr("model", model)
        trial.set_user_attr("precision_type", precision_used)

        return eval_results["eval_accuracy"]
    
    except Exception as e:
        # Log the error and return 0.0 for failed trials
        print(f"Trial {trial.number} failed with error: {str(e)}")
        traceback.print_exc()
        return 0.0

In [ ]:
import optuna

study = optuna.create_study(
    direction="maximize",
    study_name="bert-tiny-nas-study",
    sampler=sampler,
)

study.optimize(
    objective,
    n_trials=50,  # Run 50 trials to test different precision types
    timeout=60 * 60 * 24,
)

[I 2026-02-03 13:46:14,244] A new study created in memory with name: bert-tiny-nas-study
/home/ism/ADL/mase/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Traceback (most recent call last):
  File "/tmp/ipykernel_412256/3199720232.py", line 18, in objective
    trainer.train()
  File "/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/transformers/trainer.py", line 2245, in train
    return inner_training_loop(
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/transformers/trainer.py", line 2560, in _inner_training_loop
    tr_loss_step = self.training_step(model, inputs, num_items_in_batch)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/transformers/trainer.py", line 3736, in training_step
    loss = self.comput

Trial 0 failed with error: Dimension out of range (expected to be in range of [-3, 2], but got 3)


/home/ism/ADL/mase/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Traceback (most recent call last):
  File "/tmp/ipykernel_412256/3199720232.py", line 18, in objective
    trainer.train()
  File "/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/transformers/trainer.py", line 2245, in train
    return inner_training_loop(
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/transformers/trainer.py", line 2560, in _inner_training_loop
    tr_loss_step = self.training_step(model, inputs, num_items_in_batch)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/transformers/trainer.py", line 3736, in training_step
    loss = self.compute_loss(model, inputs, num_items_in_batch=num_items_in_batch)
           ^^^^^^^^^^^^^^^^^

Trial 1 failed with error: Dimension out of range (expected to be in range of [-1, 0], but got 1)


/home/ism/ADL/mase/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Traceback (most recent call last):
  File "/tmp/ipykernel_412256/3199720232.py", line 18, in objective
    trainer.train()
  File "/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/transformers/trainer.py", line 2245, in train
    return inner_training_loop(
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/transformers/trainer.py", line 2560, in _inner_training_loop
    tr_loss_step = self.training_step(model, inputs, num_items_in_batch)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/transformers/trainer.py", line 3736, in training_step
    loss = self.compute_loss(model, inputs, num_items_in_batch=num_items_in_batch)
           ^^^^^^^^^^^^^^^^^

...

Trial 26 failed with error: Dimension out of range (expected to be in range of [-3, 2], but got 3)


/home/ism/ADL/mase/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Traceback (most recent call last):
  File "/tmp/ipykernel_412256/3199720232.py", line 18, in objective
    trainer.train()
  File "/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/transformers/trainer.py", line 2245, in train
    return inner_training_loop(
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/transformers/trainer.py", line 2560, in _inner_training_loop
    tr_loss_step = self.training_step(model, inputs, num_items_in_batch)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/transformers/trainer.py", line 3736, in training_step
    loss = self.compute_loss(model, inputs, num_items_in_batch=num_items_in_batch)
           ^^^^^^^^^^^^^^^^^

Trial 27 failed with error: Dimension out of range (expected to be in range of [-3, 2], but got 3)


/home/ism/ADL/mase/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,0.536300
1000,0.474300
1500,0.476700
2000,0.482700
2500,0.462200
3000,0.473300


In [ ]:
# ============================================================================
# ANALYZE RESULTS BY PRECISION TYPE
# ============================================================================

import pandas as pd
from collections import defaultdict

# Group trials by precision type
precision_performance = defaultdict(list)

for trial in study.trials:
    if trial.value is not None and trial.value > 0:
        # Extract precision type from trial params
        precision_type = None
        for param_name in trial.params:
            if param_name.endswith('_precision_type'):
                precision_type = trial.params[param_name]
                break
        
        if precision_type:
            precision_performance[precision_type].append({
                'trial_number': trial.number,
                'accuracy': trial.value,
                'params': trial.params
            })

# Create summary statistics
summary_data = []
for precision_type, trials in precision_performance.items():
    accuracies = [t['accuracy'] for t in trials]
    summary_data.append({
        'Precision Type': precision_type,
        'Count': len(accuracies),
        'Best Accuracy': f"{max(accuracies):.4f}",
        'Mean Accuracy': f"{sum(accuracies)/len(accuracies):.4f}",
        'Worst Accuracy': f"{min(accuracies):.4f}",
    })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values('Best Accuracy', ascending=False)

print("=" * 80)
print("PRECISION TYPE PERFORMANCE SUMMARY")
print("=" * 80)
print(summary_df.to_string(index=False))
print("=" * 80)

In [ ]:
# ============================================================================
# VISUALIZE: CUMULATIVE MAX ACCURACY VS TRIAL NUMBER
# ============================================================================

import matplotlib.pyplot as plt
import numpy as np

# Get trial data
trial_numbers = []
trial_accuracies = []
cumulative_max = []

current_max = 0
for trial in study.trials:
    if trial.value is not None:
        trial_numbers.append(trial.number)
        trial_accuracies.append(trial.value)
        current_max = max(current_max, trial.value)
        cumulative_max.append(current_max)

# Create the plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Cumulative Max Accuracy
ax1.plot(trial_numbers, cumulative_max, marker='o', linewidth=2, markersize=4, color='blue', alpha=0.7)
ax1.set_xlabel('Trial Number', fontsize=12)
ax1.set_ylabel('Maximum Achieved Accuracy', fontsize=12)
ax1.set_title('Cumulative Maximum Accuracy Over Trials', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Plot 2: All trial accuracies with color by precision type
precision_colors = {
    'float32': 'red',
    'integer': 'blue',
    'minifloat_denorm': 'green',
    'minifloat_ieee': 'orange',
    'log': 'purple',
    'binary': 'brown',
    'binary_scaling': 'pink',
}

for precision_type, color in precision_colors.items():
    if precision_type in precision_performance:
        trials_data = precision_performance[precision_type]
        x = [t['trial_number'] for t in trials_data]
        y = [t['accuracy'] for t in trials_data]
        ax2.scatter(x, y, label=precision_type, color=color, alpha=0.6, s=50)

ax2.set_xlabel('Trial Number', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.set_title('Trial Accuracies by Precision Type', fontsize=14, fontweight='bold')
ax2.legend(loc='best', fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('precision_search_results.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Figure saved to 'precision_search_results.png'")